# Annotate human single-cell data with actinn-jax — with timing

This notebook annotates an **unknown human scRNA-seq dataset** using the pre-trained,
census-wide reference that ships with `actinn-jax` (~800 cell types). Everything runs on
**CPU** — no GPU, no scPRINT at inference.

We use a human **lung atlas** (Krasnow *Tabula Sapiens* 10x, 65k cells) as the query, but
just point `QUERY` at any `.h5ad` with **raw counts** (in `.X` or `.raw`) and Ensembl-ID or
symbol gene names.

**What we measure:** load time, annotation throughput (cells/s), agreement with the query's
own labels (exact *and* ontology-aware), and the effect of the abstain threshold.

## 1. Load the bundled reference (CPU, no scPRINT)

In [1]:
import time, numpy as np, pandas as pd, scanpy as sc
import actinn_jax as aj

class T:  # tiny wall-clock timer
    def __enter__(self): self.t = time.perf_counter(); return self
    def __exit__(self, *a): self.dt = time.perf_counter() - self.t

with T() as t:
    model = aj.bundled_reference('broad_human_v1')
n_groups = len(set(model.type_to_group.values()))
print(f'loaded {len(model.classes)} cell types / {n_groups} coarse groups '
      f'in {t.dt*1e3:.0f} ms')

loaded 798 cell types / 28 coarse groups in 61 ms


## 2. Load a query dataset

Swap `QUERY` for your own `.h5ad`. `use_raw='auto'` (the default) picks raw counts from
`.raw` when present.

In [2]:
QUERY = '/Users/iandriver/Downloads/krasnow_lung_atlas_10x.h5ad'  # <- your file here
with T() as t:
    adata = sc.read_h5ad(QUERY)
print(f'{adata.n_obs:,} cells x {adata.n_vars:,} genes in {t.dt:.1f}s')
print('gene ids look like:', list(adata.var_names[:2]))
print('raw counts present:', adata.raw is not None)

65,662 cells x 24,769 genes in 5.6s
gene ids look like: ['ENSG00000119048', 'ENSG00000162852']
raw counts present: True


## 3. Annotate the whole atlas — timed

`annotate` adds `celltype`, `celltype_coarse`, and `celltype_probability` to `.obs`.

In [3]:
with T() as t:
    adata = aj.annotate(adata, model)          # min_prob=None -> no abstain yet
print(f'annotated {adata.n_obs:,} cells in {t.dt:.1f}s = {adata.n_obs/t.dt:,.0f} cells/s (CPU)')
adata.obs[['celltype', 'celltype_coarse', 'celltype_probability']].head()

annotated 65,662 cells in 4.1s = 15,825 cells/s (CPU)


,celltype,celltype_coarse,celltype_probability
index,,,
P2_1_AAACCTGAGAAACCAT,pulmonary capillary endothelial cell,13,0.722690
P2_1_AAATGCCAGATGAGAG,endothelial cell,4,0.262501
P2_1_AACACGTTCGATCCCT,alveolar type 1 fibroblast cell,4,0.273194
P2_1_AACACGTTCGCACTCT,endothelial cell,4,0.180412
P2_1_AACCATGCAGCTCGCA,endothelial cell,4,0.178211


## 4. How good is it? Exact-name vs ontology-aware

The reference has ~800 **fine** Cell-Ontology types, so it often assigns a *sibling or
child* of the query's label. Exact string match punishes that harshly — e.g. `alveolar
macrophage` cells get called other macrophage subtypes:

In [4]:
true = adata.obs['cell_type'].astype(str)
pred = adata.obs['celltype'].astype(str)
print('exact-name agreement:', round((pred.values == true.values).mean(), 3))
print()
print("cells whose true label is 'alveolar macrophage' are predicted as:")
print(pred[true == 'alveolar macrophage'].value_counts().head(5).to_string())

exact-name agreement: 0.134

cells whose true label is 'alveolar macrophage' are predicted as:
celltype
alternatively activated macrophage              8487
mononuclear phagocyte                           4270
lung macrophage                                  667
metallothionein-positive alveolar macrophage     372
TREM2-positive macrophage                        310


The predictions are the right *lineage*, just finer. The honest metric is
**ontology-aware concordance**: a call counts as correct if it is the same node, an
ancestor, or a descendant of the true label in the Cell Ontology. The model stores a CL id
per class (`model.class_to_cl`), so we only need the ontology graph.

In [5]:
import os, urllib.request, pronto
OBO = '/tmp/cl-basic.obo'
if not os.path.exists(OBO):
    urllib.request.urlretrieve('http://purl.obolibrary.org/obo/cl/cl-basic.obo', OBO)
ont = pronto.Ontology(OBO)
anc = {tm.id: frozenset(s.id for s in tm.superclasses(with_self=True)) for tm in ont.terms()}

def concordant(true_cl, pred_cl, keep):
    ok = n = 0
    for tcl, pcl in zip(true_cl[keep], pred_cl[keep]):
        if not tcl or not pcl: continue
        n += 1
        ok += (tcl == pcl or pcl in anc.get(tcl, ()) or tcl in anc.get(pcl, ()))
    return ok / max(n, 1)

true_cl = adata.obs['cell_type_ontology_term_id'].astype(str).values
pred_cl = np.array([model.class_to_cl.get(p, '') for p in pred])
keep = np.array([bool(c) for c in true_cl])
print('exact-name       :', round((pred.values == true.values)[keep].mean(), 3))
print('ontology-aware   :', round(concordant(true_cl, pred_cl, keep), 3))

exact-name       : 0.134
ontology-aware   : 0.524


## 5. Abstain threshold: trade coverage for precision

`min_prob` relabels low-confidence cells `unknown` instead of force-mapping them. Because
the probabilities are fixed, we compute the frame once and threshold in NumPy (no re-run).

In [6]:
frame = model.predict_frame(adata)[0]
prob = frame['celltype_probability'].values
base = frame['celltype'].values
rows = []
for mp in [0.0, 0.3, 0.5, 0.7, 0.9]:
    lab = np.where(prob < mp, 'unknown', base)
    kept = (lab != 'unknown') & np.array([bool(c) for c in true_cl])
    pcl = np.array([model.class_to_cl.get(x, '') for x in lab])
    rows.append({'min_prob': mp, 'coverage': round(kept.mean(), 3),
                 'ontology_concordant(kept)': round(concordant(true_cl, pcl, kept), 3)})
pd.DataFrame(rows)

,min_prob,coverage,ontology_concordant(kept)
0,0.0,1.000,0.524
1,0.3,0.554,0.584
2,0.5,0.256,0.678
3,0.7,0.131,0.668
4,0.9,0.044,0.605


Higher `min_prob` → fewer cells labeled, but higher precision on the ones kept
(and novel cell types not in the reference get flagged `unknown` rather than mislabeled).
`annotate(adata, model, min_prob=0.5)` is a good default.

## 6. Build your own reference (optional, one-time)

For a narrower, higher-accuracy reference on *your* labeled data, embed it once with
scPRINT (the only GPU/Apple-MPS step; `pip install scprint`) and train:

```python
from actinn_jax.embed import scprint_embed
emb   = scprint_embed(ref_adata)                                    # GPU/MPS, once
model = aj.build_hierarchical_reference(ref_adata, 'cell_type', emb,
                                        ontology_key='cell_type_ontology_term_id')
model.save('my_reference')                                          # then annotate on CPU
```

See `examples/build_reference.py`. The accuracy/speed/memory rationale lives in the
[actinn-jax-benchmark](https://github.com/iandriver/actinn-jax-benchmark) repo.